# Branch A — BiLSTM Training
Run on GPU runtime in Google Colab.

In [ ]:
# Cell 1 — Clone repo and set path
import os, sys

if not os.path.exists('/content/hallucination-detector'):
    !git clone https://github.com/theek-sh-nahh/hallucination-detector.git
else:
    !cd /content/hallucination-detector && git pull origin main

os.chdir('/content/hallucination-detector')
sys.path.insert(0, '/content/hallucination-detector')

# Verify src is visible
print('Repo contents:',   os.listdir('/content/hallucination-detector'))
print('src contents:',    os.listdir('/content/hallucination-detector/src'))
print('sys.path[0]:',     sys.path[0])

In [ ]:
# Cell 2 — Install dependencies
!pip install -q sentence-transformers imbalanced-learn
print('Done.')

In [ ]:
# Cell 3 — Upload processed_data.zip
from google.colab import files
import zipfile, os

print('Upload processed_data.zip now:')
uploaded = files.upload()

os.makedirs('/content/hallucination-detector/data/processed', exist_ok=True)
with zipfile.ZipFile('processed_data.zip', 'r') as z:
    z.extractall('/content/hallucination-detector/data/processed')

print('Extracted files:')
print(os.listdir('/content/hallucination-detector/data/processed'))

In [ ]:
# Cell 4 — Load data and verify GPU
import numpy as np
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

base    = '/content/hallucination-detector/data/processed'
X_train = np.load(f'{base}/X_train.npy')
X_val   = np.load(f'{base}/X_val.npy')
y_train = np.load(f'{base}/y_train.npy')
y_val   = np.load(f'{base}/y_val.npy')

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'Classes: {dict(zip(*np.unique(y_train, return_counts=True)))}')

In [ ]:
# Cell 5 — SMOTE balancing
from imblearn.over_sampling import SMOTE

print('Before SMOTE:', dict(zip(*np.unique(y_train, return_counts=True))))

smote = SMOTE(
    sampling_strategy={0: 1811, 1: 1201, 2: 800, 3: 1095},
    k_neighbors=3,
    random_state=42
)
X_train, y_train = smote.fit_resample(X_train, y_train)

print('After SMOTE: ', dict(zip(*np.unique(y_train, return_counts=True))))
print(f'X_train shape: {X_train.shape}')

In [ ]:
# Cell 6 — Run hyperparameter search
from src.lstm_model import run_hyperparameter_search

save_dir = '/content/hallucination-detector/models/lstm'

best_config, results = run_hyperparameter_search(
    X_train, X_val, y_train, y_val,
    device=device,
    save_dir=save_dir
)
print(f'\nBest config: {best_config}')

In [ ]:
# Cell 7 — Train final model with best config
from src.lstm_model import (
    BiLSTMClassifier, prepare_dataloaders, train_model
)

final_config = best_config.copy()
final_config['epochs']   = 60
final_config['patience'] = 7

train_loader, val_loader = prepare_dataloaders(
    X_train, X_val, y_train, y_val,
    batch_size=final_config['batch_size']
)

model = BiLSTMClassifier(
    input_dim=384,
    hidden_dim=final_config['hidden_dim'],
    num_layers=2,
    num_classes=4,
    dropout=final_config['dropout']
).to(device)

history = train_model(
    model, train_loader, val_loader,
    final_config, device,
    save_path=f'{save_dir}/lstm_final.pt'
)
print('Final model training complete.')

In [ ]:
# Cell 8 — Plot training history
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig(f'{save_dir}/training_plot.png', dpi=150)
plt.show()
print('Plot saved.')

In [ ]:
# Cell 9 — Download trained model
import shutil
from google.colab import files

shutil.make_archive('lstm_model', 'zip',
    '/content/hallucination-detector/models/lstm')
files.download('lstm_model.zip')
print('Download started.')